# Native T4 attention benchmark
Synthetic numerical/performance check only. No model downloads, generation or credentials.

In [ ]:

import json,pathlib,time,traceback,torch
from torch.nn.functional import scaled_dot_product_attention as sdpa
from torch.nn.attention import sdpa_kernel,SDPBackend
OUT=pathlib.Path('/kaggle/working/attention-benchmark.json')
report={'stage':'started','task':'native kernel feasibility and synthetic numerical comparison','generative_quality_verified':False,'tests':[]}
def save():OUT.write_text(json.dumps(report,indent=2));print(json.dumps(report),flush=True)
save()
try:
    report.update(torch=torch.__version__,gpu=torch.cuda.get_device_name(0),capability=list(torch.cuda.get_device_capability(0)))
    torch.manual_seed(38017)
    with torch.inference_mode():
        for n,dim,amplitude in [(256,64,1),(2048,64,1),(8192,64,1),(2048,64,8),(2048,128,1)]:
            q,k,v=[(torch.randn(1,16,n,dim,device='cuda')*amplitude).bfloat16() for _ in range(3)]
            halves=[x.half() for x in (q,k,v)]
            item={'tokens':n,'heads':16,'head_dim':dim,'amplitude':amplitude,'input_max':max(x.abs().max().item() for x in (q,k,v)),'fp16_cast_finite':all(torch.isfinite(x).all().item() for x in halves),'fp16_cast_max_absolute_error':max((x.float()-y.float()).abs().max().item() for x,y in zip((q,k,v),halves))}
            results={}
            for label,inputs in [('fp32',[x.float() for x in (q,k,v)]),('fp16',halves)]:
                with sdpa_kernel(SDPBackend.EFFICIENT_ATTENTION):
                    sdpa(*inputs);torch.cuda.synchronize()
                    start=torch.cuda.Event(enable_timing=True);end=torch.cuda.Event(enable_timing=True)
                    start.record();value=sdpa(*inputs);end.record();torch.cuda.synchronize()
                item[label+'_milliseconds']=start.elapsed_time(end)
                results[label]=value.float()
            ref=results['fp32'];half=results['fp16'];delta=half-ref
            item.update(output_finite=bool(torch.isfinite(half).all()),max_absolute_error=delta.abs().max().item(),relative_rms_error=(delta.square().mean().sqrt()/ref.square().mean().sqrt().clamp_min(1e-12)).item(),bf16_output_max_error=(half.bfloat16().float()-ref.bfloat16().float()).abs().max().item(),speedup=item['fp32_milliseconds']/item['fp16_milliseconds'])
            report['tests'].append(item);save()
            del q,k,v,halves,results,ref,half,delta,value,inputs
            torch.cuda.empty_cache()
    report['linear_tests']=[]
    with torch.inference_mode():
        for tokens in [256,8192]:
            x=torch.randn(tokens,1024,device='cuda').bfloat16()
            w=(torch.randn(4096,1024,device='cuda')/32).bfloat16()
            half_x=x.half();half_w=w.half();outputs={};item={'tokens':tokens,'input_features':1024,'output_features':4096,'synthetic':True}
            for label,a,b in [('bf16',x,w),('fp16',half_x,half_w)]:
                torch.nn.functional.linear(a,b);torch.cuda.synchronize()
                start=torch.cuda.Event(enable_timing=True);end=torch.cuda.Event(enable_timing=True)
                start.record();output=torch.nn.functional.linear(a,b);end.record();torch.cuda.synchronize()
                item[label+'_milliseconds']=start.elapsed_time(end);outputs[label]=output.float()
            delta=outputs['bf16']-outputs['fp16']
            item.update(speedup=item['bf16_milliseconds']/item['fp16_milliseconds'],relative_rms_error=(delta.square().mean().sqrt()/outputs['bf16'].square().mean().sqrt()).item(),output_finite=bool(torch.isfinite(outputs['fp16']).all()))
            report['linear_tests'].append(item);save()
            del x,w,half_x,half_w,outputs,delta,output,a,b
            torch.cuda.empty_cache()
    report['stage']='synthetic_benchmark_complete';save()
except Exception as e:
    report.update(stage='benchmark_failed',error=repr(e),traceback=traceback.format_exc());save()
